# Reproducibility smoke test: 400-patient free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified, the exact script used for the published benchmark**, against the 400-patient free sample in `../sample/`.

The published benchmark was run on the full dataset (400 patients, all five regimens).  
The sample here is the same 400 patients — the published numbers are therefore exactly reproducible from this folder.

**Expected output (seed 20261119, XGBoost n\_estimators=600):**

| Feature set | Model | AUROC (mean ± SD) | AUPRC | Brier |
|---|---|---|---|---|
| characteristics + first level + microbiology | XGBoost | **0.970 ± 0.003** | 0.927 | 0.059 |
| characteristics + first level + microbiology | Logistic regression | 0.956 ± 0.002 | 0.897 | 0.069 |
| characteristics + microbiology | XGBoost | 0.893 ± 0.007 | 0.777 | 0.105 |
| characteristics + first level | XGBoost | 0.809 ± 0.005 | 0.639 | 0.136 |

Task: predict failure to keep the free beta-lactam concentration above the MIC for 100 % of hours 48–72 under regimen A (label bolus dosing), from bedside-available information.

In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, "../validation/benchmark_script.py",
     "../sample", "./smoke_test_results.parquet"],
    capture_output=True, text=True
)
print(r.stdout[-4000:])
print(r.stderr[-2000:])
assert r.returncode == 0, r.stderr

## Reading the output

- Results are grouped by `feature_set` and `model`.
- `auroc`, `auprc`, `brier` are per-fold values; the table below shows mean ± SD across the five folds.
- These numbers should match the published benchmark exactly (same 400 patients, same seed, same script).

In [ ]:
import pandas as pd
res = pd.read_parquet("./smoke_test_results.parquet")
summary = res.groupby(["feature_set", "model"])[["auroc", "auprc", "brier"]].agg(["mean", "std", "min"]).round(4)
summary